# Week 1 — Identifiers, and why they do not match

**Computer Applications in Biotechnology**

| | |
|---|---|
| **Due** | start of the Week 2 meeting |
| **Estimated time** | 2–3 hours |
| **Prerequisites** | none |

## Learning objectives

By the end of this notebook you will be able to:

1. Retrieve a protein record from UniProt programmatically and read its structure.
2. Move between UniProt, PDB, and AlphaFold DB identifiers for the same protein.
3. Distinguish experimental, homology-inferred, and automatic annotation evidence.
4. Explain why the "same" protein has several different identifiers, and what breaks when you assume it does not.

## The semester question

For fifteen weeks we are asking one thing: **given a protein, can we predict what binds to it?**

We finish by screening one drug against several hundred *E. coli* proteins and asking whether it worked. Everything between here and there is you acquiring the ability to judge that answer.

Today is the boring foundation: what a protein *is*, as far as a database is concerned.

In [ ]:
#@title Setup — run this first
%pip install -q "biopython>=1.85" "pandas>=2.0" "matplotlib>=3.7" "requests>=2.31"
print("setup complete")

In [ ]:
# Course data. 
import os, pathlib, requests

FIXTURES = pathlib.Path("fixtures")
COURSE_DATA_URL = None   

def course_file(name: str) -> pathlib.Path:
    """Return a path to a course data file, downloading it if needed."""
    local = FIXTURES / name
    if local.exists():
        return local
    if COURSE_DATA_URL:
        FIXTURES.mkdir(exist_ok=True)
        r = requests.get(f"{COURSE_DATA_URL}/{name}", timeout=30)
        r.raise_for_status()
        local.write_bytes(r.content)
        return local
    raise FileNotFoundError(
        f"{name} not found. Put the fixtures folder beside this notebook, "
        "or set COURSE_DATA_URL."
    )

print("fixtures dir:", FIXTURES.resolve())

---
## Part 1: The running example

Every demonstration this semester will use the same protein: **dihydrofolate reductase from *E. coli* K-12**, or DHFR.

It is a good choice for teaching. It is small (159 residues), it has been studied for decades, there are many experimental structures of it, and it is a real clinical drug target — the antibiotic trimethoprim works by inhibiting it. In Week 11 you will dock trimethoprim into this exact protein.

Its UniProt accession is `P0ABQ4`.

In [ ]:
import requests

ACCESSION = "P0ABQ4"

def fetch_fasta(accession: str) -> str:
    """Fetch a UniProt record in FASTA format. Returns the raw text."""
    url = f"https://rest.uniprot.org/uniprotkb/{accession}.fasta"
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    return r.text

try:
    fasta_text = fetch_fasta(ACCESSION)
    print("fetched live from UniProt")
except Exception as e:
    print(f"live fetch failed ({e}); using local fixture")
    fasta_text = course_file("dhfr_ecoli.fasta").read_text()

print(fasta_text)

The header line packs a lot in. Read it carefully:

```
>sp|P0ABQ4|DYR_ECOLI Dihydrofolate reductase OS=Escherichia coli (strain K12) OX=83333 GN=folA PE=1 SV=1
```

| Field | Meaning |
|---|---|
| `sp` | Swiss-Prot — manually reviewed. `tr` would mean TrEMBL, automatic and unreviewed. |
| `P0ABQ4` | the **accession**. Stable, machine-facing, what you should key on. |
| `DYR_ECOLI` | the **entry name**. Human-readable and *not* stable — these get renamed. |
| `OS=` / `OX=` | organism name and NCBI taxonomy identifier. |
| `GN=` | gene name. |
| `PE=1` | protein existence: 1 = evidence at protein level. |
| `SV=1` | sequence version. |

**The first trap of the semester:** `P0ABQ4` and `DYR_ECOLI` both identify this protein, but only one of them is safe to build a pipeline on. Key on accessions, always.

In [ ]:
# ---- YOUR CODE HERE ----
# Expected: a variable `sequence` holding the amino acid sequence as one string, with no header and no newlines
# Example: len(sequence) should be exactly 159, and sequence[:5] should be 'MISLI'
# ---- END ----


---
## Part 2: The same protein, three databases

A protein has one identity and many names. UniProt describes the *sequence*, the PDB holds *experimental structures*, and AlphaFold DB holds *predicted structures*.

Find your protein in all three.

In [ ]:
import json

cache = json.loads(course_file("verified_cache.json").read_text())
record = cache[ACCESSION]

print("UniProt accession :", ACCESSION)
print("entry name        :", record["entry_name"])
print("gene names        :", ", ".join(record["gene_names"]))
print("length            :", record["length"])
print("AlphaFold DB      :", record["alphafold"])
print()
print("Some PDB entries for this protein:")
for pdb_id, desc in record["pdb_examples"].items():
    print(f"  {pdb_id}  {desc}")

Notice the shape of each identifier. UniProt accessions look like `P0ABQ4`. PDB identifiers are four characters like `7NAE`. AlphaFold entries are built *from* the UniProt accession: `AF-P0ABQ4-F1`.

Look closer at that last one. AlphaFold DB is keyed on UniProt, so if you have an accession you can construct the AlphaFold identifier directly. The PDB is not keyed on UniProt; a protein may have zero structures or two hundred.

**One protein, five structures, five different experiments.** `5DFR` is the protein alone. `7DFR` has folate and NADP+ bound. `7NAE` has trimethoprim in it. These are not redundant: each captures the protein in a different state, and in Week 6 you will find that they disagree with each other in interesting places.

In [ ]:
# ---- YOUR CODE HERE ----
# Expected: a function `resolve(accession)` returning a dict with keys 'accession', 'alphafold', 'n_pdb'
# Example: resolve('P0ABQ4') should give {'accession': 'P0ABQ4', 'alphafold': 'AF-P0ABQ4-F1', 'n_pdb': 5}
# ---- END ----


---
## Part 3: Identifiers drift

`P0ABQ4` has a **secondary accession**: `P00379`. Older papers and older datasets cite it. UniProt still resolves it to the current entry.

This is the mechanism behind a large fraction of real-world data-integration bugs. Two datasets describe the same protein under two accessions; you join them; nothing matches; nothing errors.

Try it: fetch `P00379` and compare what comes back to `P0ABQ4`.

In [ ]:
try:
    old = fetch_fasta("P00379")
    old_seq = "".join(old.strip().split("\n")[1:])
    header = old.strip().split("\n")[0]
    print("Requested P00379, the header returned says:")
    print(" ", header)
    print()
    print("same sequence as P0ABQ4?", old_seq == sequence)
except Exception as e:
    print(f"live fetch unavailable ({e})")
    print("Offline: P00379 is recorded as a secondary accession of P0ABQ4 in the cache.")
    print("secondary accessions:", cache["P0ABQ4"]["secondary_accessions"])

**Stop and think.** You asked for one accession and got a record labelled with a different one. No error, no warning.

Write two sentences in the cell below: what would happen if you had 400 accessions from a 2009 paper and joined them against a table keyed on current accessions? How many rows would silently fail to match, and how would you find out?

In [ ]:
# ---- YOUR CODE HERE ----
# Expected: a markdown or comment answer, 2-3 sentences
# Example: your answer should name a specific symptom you could actually detect
# ---- END ----


---
## Part 4: How much of this is actually known?

A UniProt entry is not a list of facts. It is a list of *claims with provenance*. Some are experimental, some are inferred from similar proteins, and a great many are assigned automatically by software with no human ever looking at it.

The evidence code is more important than the annotation. "This protein binds NADPH" means something very different when it was measured than when it was guessed from a 34%-identity homolog.

Pick **five** annotation fields from the UniProt web entry for `P0ABQ4` and classify the evidence behind each.

In [ ]:
import pandas as pd

# Fill this in from https://www.uniprot.org/uniprotkb/P0ABQ4 -- expand the
# "Evidence" markers next to each annotation.
evidence = pd.DataFrame([
    # {"field": "...", "value": "...", "evidence": "experimental|homology|automatic"},
])
evidence

---
## Extensions

Complete **at least four Computing extensions and at least four Biology extensions** across the thirteen assignments. Beyond that minimum, choose freely.

### Computing extension
Write a function that takes a *list* of accessions and returns a tidy DataFrame with columns `accession`, `alphafold`, `n_pdb`, `status`. It must not raise on an accession that has no model, no structures, or no record at all — it must record what happened and continue.

### Biology extension
Take one annotation on `P0ABQ4` that traces back to a single primary paper. Read that paper's abstract. Does the annotation fairly represent what was actually shown, or has it been sharpened in transmission?

In [ ]:
# ---- YOUR CODE HERE ----
# Expected: your chosen extension(s)
# Example: label clearly which extension this is
# ---- END ----


---
## Submission checklist

- [ ] `sequence` extracted, length assertion passes
- [ ] `resolve()` handles all three test cases without raising
- [ ] Written answer on the identifier-drift question
- [ ] Evidence table with five fields classified
- [ ] At least one extension attempted and labelled
- [ ] Notebook runs top to bottom in a fresh runtime (**Runtime → Restart and run all**)

**A note on AI assistants.** You may use them. You must be able to explain any line you submit, and you must note in your submission where you used one and for what.